In [0]:
%run ./survivor_common_business

In [0]:
def generate_source_system_json(consumer_df):   
    # 读consumer表
    sconsumer_df = consumer_df \
    .select("consumermdmkey", "scon_srcs_code", "scon_sourcetimestamp", "scon_mrkt_code", 
            "scon_dvsn_code", "scon_brnd_code", "scon_consumerid", "scon_aff_code", 
            F.lit(None).alias("TerminalId"),
            (F.coalesce(F.col("scon_srcc_action"), F.lit("")) == "DELETE").alias("DeleteFlag")) \
    .distinct() \
    .orderBy("scon_mrkt_code","consumermdmkey", "scon_srcs_code", "scon_sourcetimestamp", "scon_dvsn_code", "scon_brnd_code", "scon_consumerid", "scon_aff_code")

    # 构建SourceSystem结构体
    source_system_struct = F.struct(
        F.col("scon_srcs_code").alias("@Code"),
        F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scon_aff_code").alias("AffiliateCode"),
        F.col("scon_mrkt_code").alias("MarketCode"),
        F.col("scon_dvsn_code").alias("DivisionCode"),
        F.col("scon_brnd_code").alias("BrandCode"),
        F.col("scon_consumerid").alias("ConsumerId"),
        F.col("TerminalId").alias("TerminalId"),
        F.col("DeleteFlag").alias("DeleteFlag")
    )
    
    # 生成最终DataFrame
    final_df = (sconsumer_df
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg( F.collect_list(source_system_struct).alias("item_list"))
        .select(
            F.col("scon_mrkt_code"),
            F.col("consumermdmkey"),
            F.to_json( F.struct(F.col("item_list").alias("SourceSystem")), options={"ignoreNullFields": "false"}).alias("SourceSystemJSON")
        ))

    return final_df

In [0]:
def generate_attributes_json(consumer_df, attributes_table):
    # 读consumer表
    sconsumer_df =consumer_df.select("scon_mrkt_code", "consumermdmkey", "scon_id").distinct()

    # 读attributes表
    attributes_df = spark.table(attributes_table).select("sccu_id", "sccu_mrkt_code", "sccu_scon_id", "sccu_name", "sccu_value")

    cond = ((sconsumer_df.scon_id == attributes_df.sccu_scon_id) &
            (sconsumer_df.scon_mrkt_code == attributes_df.sccu_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(attributes_df, cond, "inner") \
        .select(
            "scon_mrkt_code",
            "consumermdmkey",
            "sccu_name",
            "sccu_value"
        )

    dedup_df = (base_df
        .filter(F.col("sccu_name").isin(attributes_deduplication_key))
        .groupBy("scon_mrkt_code","consumermdmkey","sccu_name")
        .agg(F.max("sccu_value").alias("sccu_value"))
    )

    other_df = base_df.filter(~(F.col("sccu_name").isin(attributes_deduplication_key)))

    all_df = dedup_df.unionByName(other_df)

    
    # 构建CustomAttributes结构体
    attributes_struct = F.struct(
        F.col("sccu_name").alias("@Name"),
        F.col("sccu_value").alias("@Value")
    )
    
    # 生成最终DataFrame
    final_df = (all_df
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(F.collect_list(attributes_struct).alias("item_list"))
        .select(
            F.col("scon_mrkt_code"),
            F.col("consumermdmkey"),
            F.to_json(F.struct(F.col("item_list").alias("CustomAttribute")), options={"ignoreNullFields": "false"}).alias("CustomAttributesJSON")
        ))

    return final_df

In [0]:
def generate_program_json(consumer_df, program_table, dim_cbr_exclude_program_table):
    # 读consumer表
    consumer_df = consumer_df.select("scon_mrkt_code", "consumermdmkey", "scon_id").distinct()

    # 读program表
    program_df = spark.table(program_table) \
        .select("scpr_id", "scpr_mrkt_code", "scpr_scon_id", "scpr_application_toch_code", "scpr_consumer_grp", 
        "scpr_prgt_code", "scpr_prgt_desc", "scpr_prgl_code", "scpr_prgl_desc", 
        "scpr_system_code", "scpr_system_desc", "scpr_membershipnum", "scpr_cardnum", 
        "scpr_start_dt", "scpr_end_dt", "scpr_acquiredpoint_num", "scpr_redeemedpoint_num", 
        "scpr_initial_quota", "scpr_available_quota")
    
    cond = ((consumer_df.scon_id == program_df.scpr_scon_id) &
            (consumer_df.scon_mrkt_code == program_df.scpr_mrkt_code)
           )
    
    # 读取program排除表
    exclude_program_df = spark.table(dim_cbr_exclude_program_table) \
        .select("MarketCode", "prgt_code")

    # 关联两张表
    base_df = (consumer_df
        .join(program_df,cond,"inner")
        .join(exclude_program_df, (program_df.scpr_mrkt_code == exclude_program_df.MarketCode) & (program_df.scpr_prgt_code == exclude_program_df.prgt_code) ,"left_anti")
        .select(
            "scon_mrkt_code", "scpr_id", "consumermdmkey", "scpr_scon_id", "scpr_application_toch_code", 
            "scpr_consumer_grp", "scpr_prgt_code", "scpr_prgt_desc", "scpr_prgl_code", "scpr_prgl_desc", 
            "scpr_system_code", "scpr_system_desc", "scpr_membershipnum", "scpr_cardnum", "scpr_start_dt", 
            "scpr_end_dt", "scpr_acquiredpoint_num", "scpr_redeemedpoint_num", "scpr_initial_quota", 
            "scpr_available_quota"
        )
        .distinct()
    )
    # .orderBy("scon_mrkt_code","consumermdmkey", "scpr_id")

    # 构建Program结构体
    program_struct = F.struct(
        F.col("scpr_application_toch_code").alias("ApplicationTouchPointCode"),
        F.col("scpr_consumer_grp").alias("ConsumerGroup"),
        F.col("scpr_prgt_code").alias("ProgramTypeCode"),
        F.col("scpr_prgt_desc").alias("ProgramTypeDescription"),
        F.col("scpr_prgl_code").alias("ProgramLevelCode"),
        F.col("scpr_prgl_desc").alias("ProgramLevelDescription"),
        F.col("scpr_system_code").alias("ProgramSystemIDCode"),
        F.col("scpr_system_desc").alias("ProgramSystemIDDescription"),
        F.col("scpr_membershipnum").alias("MembershipNum"),
        F.col("scpr_cardnum").alias("CardNum"),
        F.date_format(F.col("scpr_start_dt"), timestamp_format).alias("StartTimestamp"),
        F.date_format(F.col("scpr_end_dt"), timestamp_format).alias("EndTimestamp"),
        F.col("scpr_acquiredpoint_num").alias("PointsAcquired"),
        F.col("scpr_redeemedpoint_num").alias("PointsRedeemed"),
        F.col("scpr_initial_quota").alias("InitialQuota"),
        F.col("scpr_available_quota").alias("AvailableQuota")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(F.collect_list(program_struct).alias("item_list"))
        .select(
            F.col("scon_mrkt_code"),
            F.col("consumermdmkey"),
            F.to_json(F.struct(F.col("item_list").alias("Program")) , options={"ignoreNullFields": "false"}).alias("ProgramJSON")
        ))

    return final_df

In [0]:
def generate_action(consumer_df):
    final_df = (consumer_df
        .groupBy("SCON_MRKT_CODE", "consumermdmkey")
        .agg(
            F.when(F.count(F.when(F.coalesce(F.col("scon_srcc_action"), F.lit(""))  != "DELETE", 1)) > 0,  "CREATE").otherwise("DELETE").alias("header_action")
        ))
    
    return final_df

In [0]:
def fetch_cbr_records_l1(consumer_df, overall_maxtimestamp_df, group_cols):
    
    window_core = Window.partitionBy(*group_cols)

    # 1. 基础数据读取
    base_df = consumer_df.join(overall_maxtimestamp_df, group_cols, "inner") \
        .select(
            "consumermdmkey", "scon_id", "scon_srcs_code", "scon_sourcetimestamp",
            "scon_gndr_code", "scon_birthday", "scon_birthmonth", "scon_birthyear",
            "scon_reg_dt", "scon_registration_toch_code", "scon_registration_prsn_code",
            "scon_aff_code", "scon_mrkt_code", "scon_dvsn_code", "scon_brnd_code",
            "scon_wlng_code", "scon_slng_code", "scon_cntr_isoalpha3code", "scon_cvls_code",
            "scon_englishname_quality_code", "scon_localname_quality_code", "scon_localname2_quality_code",
            "scon_englishfirstname", "scon_englishmiddlename", "scon_englishlastname", "scon_englishfullname",
            "scon_localfirstname", "scon_locallastname", "scon_localfullname",
            "scon_localfirstname2", "scon_locallastname2", "scon_localfullname2",
            "scon_delete_flag", "maxoveralltimestamp_contact", "scon_consumerid", "scon_clas_code","scon_update_dt") \

    # 2. 窗口计算
    # is_latest_source：max scon_sourcetimestamp (aus原sql没有这个逻辑，现代码与其他market保持一致)
    # is_earliest_reg：min scon_reg_dt (twn计算minregdate时特殊逻辑)
    # is_latest_birth：max scon_sourcetimestamp by birth info
    base_df = base_df \
        .withColumn("maxsourcetimestamp",F.coalesce(
            F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scon_sourcetimestamp"))).over(window_core),
            F.max(F.col("scon_sourcetimestamp")).over(window_core) )) \
        .withColumn("dense_rank_cid", F.dense_rank().over(window_core.orderBy("scon_consumerid"))) \
        .withColumn("distinct_consumerid_count", F.max("dense_rank_cid").over(window_core)) \
        .withColumn("has_onlshell", F.max(F.when(F.coalesce(F.col("scon_clas_code"), F.lit("")) == "onlshell", 1).otherwise(0)).over(window_core) > 0) \
        .withColumn("minregdate",
            F.when(
            # TWN特殊逻辑：count(distinct scon_consumerid)>1 and scon_clas_code=onlshell取max scon_reg_dt，其他情况取min scon_reg_dt
            (F.col("scon_mrkt_code") == "TWN") & (F.col("distinct_consumerid_count") > 1) & F.col("has_onlshell"), 
            F.max("scon_reg_dt").over(window_core)
            ).otherwise(F.min("scon_reg_dt").over(window_core)) ) \
        .withColumn("maxbirthtimestamp", 
            F.coalesce(
                # 优先级1：有 birthyear 且未删除
                F.max(F.when(
                    (F.coalesce(F.col("scon_birthyear"), F.lit("")) != "") & 
                    (F.col("scon_delete_flag") != 1),
                    F.col("scon_sourcetimestamp")
                )).over(window_core),
                # 优先级2：任意生日字段非空且未删除
                F.max(F.when(
                    (F.concat_ws("", F.col("scon_birthday"), F.col("scon_birthmonth"), F.col("scon_birthyear")) != "") &
                    (F.col("scon_delete_flag") != 1),
                    F.col("scon_sourcetimestamp")
                )).over(window_core),
                # 优先级3：有 birthyear（忽略删除标志）
                F.max(F.when(
                    F.coalesce(F.col("scon_birthyear"), F.lit("")) != "",
                    F.col("scon_sourcetimestamp")
                )).over(window_core),
                # 优先级4：任意生日字段非空（忽略删除标志）
                F.max(F.when(
                    F.concat_ws("", F.col("scon_birthday"), F.col("scon_birthmonth"), F.col("scon_birthyear")) != "",
                    F.col("scon_sourcetimestamp")
                )).over(window_core),
                # 默认：整体最大时间戳
                F.max("scon_sourcetimestamp").over(window_core)
            )) \
        .withColumn("is_latest_source", F.col("scon_sourcetimestamp") == F.col("maxsourcetimestamp")) \
        .withColumn("is_earliest_reg", F.col("scon_reg_dt") == F.col("minregdate")) \
        .withColumn("is_latest_birth", F.col("scon_sourcetimestamp") == F.col("maxbirthtimestamp"))

    # 3.1 最新sourcetimestamp记录（别名a）
    latest_overall_df = base_df.filter(F.col("is_latest_source"))
    latest_overall_addflag_df = add_namefilledflag_by_market(latest_overall_df) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_update_dt").desc(),F.col("scon_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # 3.2. 最早reg_dt记录（别名b）
    earliest_reg_df = base_df.filter(F.col("is_earliest_reg")) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_sourcetimestamp"), F.col("scon_update_dt"),F.col("scon_id")))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # 3.3 最新生日记录（别名d）
    latest_birth_df = base_df.filter(F.col("is_latest_birth")) \
        .select(
            F.col("consumermdmkey"),
            F.col("scon_brnd_code"),
            F.col("scon_mrkt_code"),
            F.col("scon_birthday"),
            F.col("scon_birthmonth"),
            F.col("scon_birthyear"),
            F.col("scon_update_dt"),
            F.col("scon_id")
        ) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_birthyear").desc(), F.col("scon_birthmonth").desc(), F.col("scon_update_dt").desc(), F.col("scon_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")
    
    # 4. 最终关联
    final_df = latest_overall_addflag_df.alias("a") \
        .join(earliest_reg_df.alias("b"), group_cols, "inner") \
        .join(latest_birth_df.alias("d"), group_cols, "inner") \
        .select(
            F.col("a.consumermdmkey"),
            F.col("a.scon_aff_code"),
            F.col("a.scon_mrkt_code"),
            F.col("a.scon_dvsn_code"),
            F.col("a.scon_brnd_code"),
            F.col("a.scon_srcs_code"),
            F.col("a.maxoveralltimestamp_contact").alias("scon_sourcetimestamp"),
            F.col("a.scon_gndr_code"),
            F.col("d.scon_birthday"),
            F.col("d.scon_birthmonth"),
            F.col("d.scon_birthyear"),
            F.col("b.scon_reg_dt"),
            F.col("b.scon_sourcetimestamp").alias("scon_reg_sourcetimestamp"),
            F.col("b.scon_registration_toch_code"),
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_srcs_code")).alias("scon_prs_sourcesystemcode"),       
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_sourcetimestamp")).alias("scon_prs_sourcetimestamp"),       
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_aff_code")).alias("scon_prs_aff_code"),       
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_mrkt_code")).alias("scon_prs_mrkt_code"),       
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_dvsn_code")).alias("scon_prs_dvsn_code"),       
            F.when(F.coalesce(F.col("b.scon_registration_prsn_code"), F.lit("")) != "", F.col("b.scon_brnd_code")).alias("scon_prs_brnd_code"),
            F.col("b.scon_registration_prsn_code"),
            F.col("a.scon_registration_toch_code").alias("scon_lastupdate_toch_code"),
            F.col("a.scon_wlng_code"),
            F.col("a.scon_slng_code"),
            F.col("a.scon_cntr_isoalpha3code"),
            F.col("a.scon_cvls_code"),
            F.col("a.NameFilledFlag"),
            F.col("a.scon_id"),
            F.col("a.scon_update_dt")
        ) \
        # .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_update_dt").desc(),F.col("scon_id").desc()))) \
        # .filter(F.col("rn") == 1) \
        # .drop("rn","scon_id","scon_update_dt")

    return final_df

In [0]:
def process_consumer_l1_data(consumer_join_sources_df, task_id, group_cols):      
    media_df = spark.table(media_table)
    phone_df = spark.table(phone_table)
    address_df = spark.table(address_table)
    optin_df = spark.table(optin_table)
     
    # 排除acs
    consumer_noacs_df = consumer_join_sources_df.filter(F.col("is_acs_source") != 1)

    # 1.  person 计算: 排除ACS
    # 1.1 twn 还需进行 onlshell 过滤
    twn_df = consumer_noacs_df.filter(F.col("scon_mrkt_code") == "TWN")
    others_df = consumer_noacs_df.filter(~F.col("scon_mrkt_code").isin(["TWN"]))
    consumer_by_filter_df = others_df.unionByName(filter_twn_data(twn_df, group_cols))

    overall_maxtimestamp_df = calculate_overall_maxtimestamp(consumer_by_filter_df, media_df, phone_df, address_df, optin_df, group_cols)
    cbr_records_df = fetch_cbr_records_l1(consumer_by_filter_df, overall_maxtimestamp_df, group_cols)
    
    # 2. sourcesystem 计算: 排除ACS
    source_system_df = generate_source_system_json(consumer_noacs_df)

    # 3&4 program, customattribute 计算
    attributes_df = generate_attributes_json(consumer_join_sources_df, attributes_table)
    program_df = generate_program_json(consumer_join_sources_df, program_table, dim_cbr_exclude_program_table)

    # 4. header_action(databricks 新增字段, 用于cbr header json生成)
    action_df = generate_action(consumer_noacs_df)
    
    result_df = cbr_records_df.join(source_system_df, group_cols, "left") \
        .join(attributes_df, group_cols, "left") \
        .join(program_df, group_cols, "left") \
        .join(action_df, group_cols, "left") \
        .withColumn("UPDATE_DT", F.current_timestamp()) \
        .withColumn("UPDATE_UID", F.lit(None)) \
        .withColumn("TASK_ID", F.lit(task_id)) \
        .withColumn("RecordTimeStamp",F.current_timestamp()) \
        .withColumn("RecordUUID",F.expr("uuid()")
    )
    
    # display(result_df)
    
    table_name = f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l1"
    print(f"table_name: {table_name}")
    merge_condition = get_merge_condition(group_cols)
    merge_t_table(table_name, result_df, merge_condition)


In [0]:
task_id = dbutils.widgets.get("task_id")
print("task_id: "+task_id)

cid_limit_num = int(get_ex_param("cid_limit_num", "50"))
print(f"cid_limit_num: {cid_limit_num}")

cid_limit_markets = get_ex_param("cid_limit_markets", "AUS,HKG,IDN,JPN,KOR,MYS,NZL,PHL,SGP,THA,TWN,VNM").split(",")
print(f"cid_limit_markets: {cid_limit_markets}")

with StepLogger("t_derived_consumer_l1", "06-1", "consumerlist", task_id=task_id) as logger:
    # 计算的维度
    group_cols = ["consumermdmkey", "scon_mrkt_code"]
    attributes_deduplication_key = ["Derived_Location", "Derived_City", "Derived_Province", "South_Chinese_Flag"]

    # 取当前task_id下的所有的uid
    consumermdmkey_df = spark.table(consumer_table).filter(F.col("task_id") == task_id).select(*group_cols).distinct()
    exclude_sources_df = spark.table(dim_excludesource_table)

    # ukey排除表（marketcode + consumermdmkey），配置需要排除的ukey，后续在关联时进行过滤
    exclude_ukey_df = spark.table(survive_exclude_ukey_table) \
        .select(F.col("marketcode").alias("exclude_marketcode"), F.col("consumermdmkey").alias("exclude_consumermdmkey")) \
        .distinct()

    # high count ukey  (scon_mrkt_code + consumermdmkey), 需要排除cid过多的ukey
    high_count_ukey_df = get_high_count_ukey(cid_limit_num, cid_limit_markets)

    # 取uid下所有的cid，并过滤关联source表
    consumer_join_sources_df = spark.table(consumer_table) \
            .join(consumermdmkey_df, group_cols, "inner") \
            .join(
                F.broadcast(exclude_ukey_df),
                (F.col("scon_mrkt_code") == F.col("exclude_marketcode")) &
                (F.col("consumermdmkey") == F.col("exclude_consumermdmkey")),
                "left_anti"
            ) \
            .join(
                F.broadcast(high_count_ukey_df),
                ["scon_mrkt_code", "consumermdmkey"],
                "left_anti"
            ) \
            .join(F.broadcast(exclude_sources_df),
                (F.col("scon_mrkt_code") == F.col("tmec_marketcode")) &
                (F.col("scon_srcs_code") == F.col("tmec_sourcesystemcode")),
                "left") \
            .withColumn("is_acs_source", F.when(F.col("tmec_marketcode").isNotNull() & (F.col("tmec_type") == "ACS"), 1).otherwise(0)) \
            .drop("tmec_id","tmec_marketcode","tmec_type","tmec_sourcesystemcode")

    # 检查是否有数据，没有则直接结束；有数据再执行process_consumer_l1_data
    if consumer_join_sources_df.isEmpty():
        print("consumer_join_sources_df 无数据，跳过后续处理")
    else:
        print(f"consumer_join_sources_df count: {consumer_join_sources_df.count()}")
        process_consumer_l1_data(consumer_join_sources_df, task_id, group_cols)